In [1]:
# autorelead libraries
%load_ext autoreload
%autoreload 2

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("torch sees:", torch.cuda.device_count(), "GPUs")
print("Current device index:", torch.cuda.current_device())
print("Device name:", torch.cuda.get_device_name())


Using: cuda
CUDA_VISIBLE_DEVICES: 3
torch sees: 1 GPUs
Current device index: 0
Device name: NVIDIA GeForce RTX 2080 Ti


In [2]:
import os
import time
import random
import pickle
import argparse
import heartpy as hp
from heartpy.datautils import rolling_mean
import numpy as np
import scipy.sparse as sp
import scipy.signal as sig
import scipy.stats as stats
import matplotlib.pyplot as plt
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader

from config import get_config
from dataset import data_loader
from neural_methods.model.ContrastPhys import ContrastPhys
from neural_methods.model.PhysNet import PhysNet_padding_Encoder_Decoder_MAX
from neural_methods.model.FactorizePhys.FactorizePhys import FactorizePhys
from neural_methods.model.FactorizePhys.FactorizePhysBig import FactorizePhysBig
from neural_methods.model.PhysMamba import PhysMamba
from neural_methods.model.ContrastFusion import ContrastFusion
from neural_methods.model.PhysFormer import ViT_ST_ST_Compact3_TDC_gra_sharp
from neural_methods.model.RhythmFormer import RhythmFormer

2025-07-28 19:10:26.668358: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-07-28 19:10:26.684429: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1753747826.702276   22359 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1753747826.707575   22359 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1753747826.721195   22359 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [3]:
def calcMetrics(input_data, sampFreq, B, window_size=None, use_harmonic=False, normalize=False):
    # define goodness parameters
    B1 = 0.75 # low cutoff
    B2 = 3 # high cutoff

    if window_size is None:
        pulseCalcLength = len(input_data)
    else:
        pulseCalcLength = np.uint32(window_size/(1.0/sampFreq))
    pulseRate_result = np.empty(len(input_data)-pulseCalcLength + 1)
    goodnessMetric_result = np.empty(len(input_data)-pulseCalcLength + 1)

    for i in range(0, len(pulseRate_result)):
        window = input_data[i:i+pulseCalcLength]
        # sigFreq, sigPower = sig.periodogram(window, fs=sampFreq, nfft=180000)
        sigFreq, sigPower = sig.welch(x=window, nperseg=len(window)//3, fs=sampFreq, nfft=180000)
        maskFreq = (sigFreq > B1)&(sigFreq<B2)
        # sigFreq = sigFreq[maskFreq]
        sigPower = sigPower * maskFreq
        if use_harmonic:
            # harmonic PSD
            harmonicsigPower = sigPower[::2]
            harmonicsigPower = np.pad(harmonicsigPower, (0, len(sigPower) - len(harmonicsigPower)))
            # find peak frequency
            peakFreq = sigFreq[np.argmax(sigPower + harmonicsigPower)]
        else:
            peakFreq = sigFreq[np.argmax(sigPower)]
        pulseRate = 60.0*peakFreq
        pulseRate_result[i] = pulseRate
        # compute goodness
        aroundPulseRate = (sigFreq > peakFreq - B) & (sigFreq < peakFreq + B)
        withinBandpass = (sigFreq >= B1) & (sigFreq <= B2)
        powerPulseRate = np.sum(sigPower[aroundPulseRate])
        powerAll = np.sum(sigPower[withinBandpass])
        if normalize:
            goodnessMetric_result[i] = powerPulseRate / powerAll
        else:
            goodnessMetric_result[i] = powerPulseRate / (powerAll - powerPulseRate)
    timestamps = np.arange(len(pulseRate_result))
    return timestamps, goodnessMetric_result, pulseRate_result

def custom_detrend(sig, Lambda):
    """custom_detrend(sig, Lambda) -> filtered_signal
    This function applies a detrending filter.
    This code is based on the following article "An advanced detrending method with application
    to HRV analysis". Tarvainen et al., IEEE Trans on Biomedical Engineering, 2002.
    *Parameters*
      ``sig`` (1d numpy array):
        The sig where you want to remove the trend.
      ``Lambda`` (int):
        The smoothing parameter.
    *Returns*
      ``filtered_signal`` (1d numpy array):
        The detrended sig.
    """
    signal_length = sig.shape[0]

    # observation matrix
    H = np.identity(signal_length)

    # second-order difference matrix

    ones = np.ones(signal_length)
    minus_twos = -2 * np.ones(signal_length)
    diags_data = np.array([ones, minus_twos, ones])
    diags_index = np.array([0, 1, 2])
    D = sp.spdiags(diags_data, diags_index, (signal_length - 2), signal_length).toarray()
    filtered_signal = np.dot((H - np.linalg.inv(H + (Lambda ** 2) * np.dot(D.T, D))), sig)
    return filtered_signal

def pulse_rate_from_power_spectral_density(pleth_sig: np.array, FS: float,
                                           LL_PR: float, UL_PR: float,
                                           BUTTER_ORDER: int = 6,
                                           DETREND: bool = False,
                                           FResBPM: float = 0.1,
                                           HARMONIC: bool = False,
                                           WELCH = True) -> float:
    """ Function to estimate the pulse rate from the power spectral density of the plethysmography sig.

    Args:
        pleth_sig (np.array): Plethysmography sig.
        FS (float): Sampling frequency.
        LL_PR (float): Lower cutoff frequency for the butterworth filtering.
        UL_PR (float): Upper cutoff frequency for the butterworth filtering.
        BUTTER_ORDER (int, optional): Order of the butterworth filter. Give None to skip filtering. Defaults to 6.
        DETREND (bool, optional): Boolena Flag for executing cutsom_detrend. Defaults to False.
        FResBPM (float, optional): Frequency resolution. Defaults to 0.1.

    Returns:
        pulse_rate (float): _description_
    

    Daniel McDuff, Ethan Blackford, January 2019
    Copyright (c)
    Licensed under the MIT License and the RAIL AI License.
    """

    N = (60*FS)/FResBPM

    # Detrending + nth order butterworth + periodogram
    if DETREND:
        pleth_sig = custom_detrend(pleth_sig, 100)
    if BUTTER_ORDER:
        [b, a] = sig.butter(BUTTER_ORDER, [LL_PR/60, UL_PR/60], btype='bandpass', fs = FS)
    pleth_sig = sig.filtfilt(b, a, np.double(pleth_sig))
    
    # Calculate the PSD and the mask for the desired range
    if WELCH:
        F, Pxx = sig.welch(x=pleth_sig, nperseg=len(pleth_sig)//3, nfft=N, fs=FS)
    else:
        F, Pxx = sig.periodogram(x=pleth_sig,  nfft=N, fs=FS);  
    FMask = (F >= (LL_PR/60)) & (F <= (UL_PR/60))
    
    # Calculate predicted pulse rate:
    FRange = F * FMask
    PRange = Pxx * FMask

    if HARMONIC:
      harmonicsigPower = PRange[::2]
      harmonicsigPower = np.pad(harmonicsigPower, (0, len(PRange) - len(harmonicsigPower)))
      harmonicsigPower[0] = 0
      MaxInd = np.argmax(PRange+harmonicsigPower)
    else:
      MaxInd = np.argmax(PRange)
    pulse_rate_freq = FRange[MaxInd]
    pulse_rate = pulse_rate_freq*60
            
    return pulse_rate


def get_ibi(pred, gt, fs=30, smooth_window=7):
    if smooth_window:
        pred = np.convolve(pred, np.ones((smooth_window))/smooth_window, mode='same')
        gt = np.convolve(gt, np.ones((smooth_window))/smooth_window, mode='same')
    rol_mean = rolling_mean(pred, windowsize=1, sample_rate = fs)
    wd = hp.peakdetection.detect_peaks(pred, rol_mean, ma_perc = 20, sample_rate = fs)
    pred_peaks = wd['peaklist']
    # plt.figure(figsize=(12, 6))
    # plt.plot(pred, label='Filtered and Smoothed rPPG Signal', color='b')
    # plt.plot(pred_peaks, pred[pred_peaks], 'rx', label='Detected Peaks', markersize=10)
    # plt.legend()
    # plt.grid(True)

    rol_mean = rolling_mean(gt, windowsize=1, sample_rate = fs)
    wd = hp.peakdetection.detect_peaks(gt, rol_mean, ma_perc = 20, sample_rate = fs)
    gt_peaks = wd['peaklist']
    # plt.figure(figsize=(12, 6))
    # plt.plot(gt, label='Filtered and Smoothed rPPG Signal', color='b')
    # plt.plot(gt_peaks, gt[gt_peaks], 'rx', label='Detected Peaks', markersize=10)
    # plt.legend()
    # plt.grid(True)
    pred_ibi = np.diff(pred_peaks)
    gt_ibi = np.diff(gt_peaks)
    return pred_ibi, gt_ibi

def get_filtered_ibi(pred, gt, fs=30, smooth_window=7):
    pred_ibi, gt_ibi = get_ibi(pred, gt, fs=fs, smooth_window=smooth_window)
    # Filter IBI (read the code above)
    pred_q1 = np.percentile(pred_ibi, 25)
    pred_q2 = np.percentile(pred_ibi, 50)
    pred_q3 = np.percentile(pred_ibi, 75)
    pred_iqr = pred_q3 - pred_q1
    pred_lower_bound = pred_q1 - 1.5 * pred_iqr
    pred_upper_bound = pred_q3 + 1.5 * pred_iqr
    pred_filtered_ibi = [x for x in pred_ibi if pred_lower_bound <= x <= pred_upper_bound]
    gt_q1 = np.percentile(gt_ibi, 25)
    gt_q2 = np.percentile(gt_ibi, 50)
    gt_q3 = np.percentile(gt_ibi, 75)
    gt_iqr = gt_q3 - gt_q1
    gt_lower_bound = gt_q1 - 1.5 * gt_iqr
    gt_upper_bound = gt_q3 + 1.5 * gt_iqr
    gt_filtered_ibi = [x for x in gt_ibi if gt_lower_bound <= x <= gt_upper_bound]
    # Sample to secs
    pred_filtered_ibi = np.array(pred_filtered_ibi) * (1/fs)
    gt_filtered_ibi = np.array(gt_filtered_ibi) * (1/fs)
    return pred_filtered_ibi, gt_filtered_ibi

In [4]:
class Args:
    # config_file = 'configs/train_configs/CogPhys_CONTRASTPHYS_BASIC.yaml'
    config_file = 'configs/train_configs/CogPhys_Fusion_BASIC.yaml'
    cached_path = None
    preprocess = None
    lr = None
    model_file_name = None

args = Args()
config = get_config(args)
# print('Configuration:')
# print(config, end='\n\n')
data_loader_dict = dict() # dictionary of data loaders 
train_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
train_data_loader = train_loader(
    name="train",
    data_path=config.TRAIN.DATA.DATA_PATH,
    config_data=config.TRAIN.DATA,
    device=config.DEVICE)
data_loader_dict['train'] = DataLoader(
    dataset=train_data_loader,
    num_workers=2,
    batch_size=2,
    shuffle=True,
)
print(); print()

valid_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
valid_data_loader = valid_loader(
    name="valid",
    data_path=config.VALID.DATA.DATA_PATH,
    config_data=config.VALID.DATA,
    device=config.DEVICE)
data_loader_dict['valid'] = DataLoader(
    dataset=valid_data_loader,
    num_workers=2,
    batch_size=2,
    shuffle=True,
)
print(); print()

test_loader = data_loader.CogPhysLoader.CogPhysLoader
print(config.DEVICE)
test_data_loader = test_loader(
    name="test",
    data_path=config.TEST.DATA.DATA_PATH,
    config_data=config.TEST.DATA,
    device=config.DEVICE)
data_loader_dict['test'] = DataLoader(
    dataset=test_data_loader,
    num_workers=4,
    batch_size=3,
    shuffle=False,
)

=> Merging a config file from configs/train_configs/CogPhys_Fusion_BASIC.yaml
cuda
Excluding ['v19_still'] files from the dataset due to corrupted nir video
Cached Data Path /shared/ab227/CogPhys/chunked_dataset

Data Path /shared/ab227/CogPhys/chunked_dataset

train Preprocessed Dataset Length: 1764



cuda
Excluding ['v19_still'] files from the dataset due to corrupted nir video
Cached Data Path /shared/ab227/CogPhys/chunked_dataset

Data Path /shared/ab227/CogPhys/chunked_dataset

valid Preprocessed Dataset Length: 144



cuda
Excluding ['v19_still'] files from the dataset due to corrupted nir video
Cached Data Path /shared/ab227/CogPhys/chunked_dataset

Data Path /shared/ab227/CogPhys/chunked_dataset

test Preprocessed Dataset Length: 720



In [15]:
test_data_loader.input_preproc, test_data_loader.label_preproc, test_data_loader.input_keys, test_data_loader.label_keys

([['NormAndFloat'], ['NormAndFloat']],
 [['Downsample', 'Standardize']],
 ['rgb_left', 'nir'],
 ['ppg'])

In [16]:
valid_data_loader.input_preproc, valid_data_loader.label_preproc, valid_data_loader.input_keys, valid_data_loader.label_keys

([['NormAndFloat'], ['NormAndFloat']],
 [['Downsample', 'Standardize']],
 ['rgb_left', 'nir'],
 ['ppg'])

In [6]:
torch.cuda.empty_cache()

In [7]:
#all_epochs_load_path = '/home/ab227/CogPhys/runs/exp/fusion_rppg_early_mid_pre_nir/PreTrainedModels'
#CogPhys_rPPG_ch3_PhysNet_Epoch49.pth

def load_one_path(all_epochs_load_path, epoch_num):
    load_path = os.path.join(all_epochs_load_path, f'CogPhys_rPPG_ch3_PhysNet_Epoch{epoch_num}.pth') #change second part based on model used
    
    if config.MODEL.NAME == 'ContrastPhys':
        model = ContrastPhys(S=config.MODEL.CONTRASTPHYS.S).to(config.DEVICE).eval()
    elif config.MODEL.NAME == 'ContrastFusion':
        model = ContrastFusion(S=config.MODEL.CONTRASTFUSION.S).to(config.DEVICE).eval()
        model = torch.nn.DataParallel(model, device_ids=list(range(1)))
    else:
        raise NotImplementedError(f"Model {config.MODEL.NAME} not implemented.")
    model.load_state_dict(torch.load(load_path, map_location=config.DEVICE), strict=True)
    
    return model

In [8]:
def forward_pass(model, img_data, gra_sharp=2.0):
    if config.MODEL.NAME == 'ContrastPhys':
        out = model(img_data)[0][-1].squeeze(0).cpu().numpy()
    elif config.MODEL.NAME == 'ContrastFusion':
        out = model(img_data)[0][-1].squeeze(0).cpu().numpy()
    elif config.MODEL.NAME == 'Physnet':
        out = model(img_data)[0].squeeze(0).cpu().numpy()
    elif config.MODEL.NAME == 'PhysMamba':
        out = model(img_data).squeeze(0).cpu().numpy()
    elif config.MODEL.NAME == 'FactorizePhys':
        out = model(img_data)[0].squeeze(0).cpu().numpy()
    elif config.MODEL.NAME == 'PhysFormer':
        out = model(img_data, gra_sharp)[0][-1].squeeze(0).cpu().numpy()
    elif config.MODEL.NAME == 'RhythmFormer':
        out = model(img_data).squeeze(0).cpu().numpy()
    else:
        raise NotImplementedError(f"Model {config.MODEL.NAME} not implemented.")
    return out

In [9]:
def process_hr_ibi(
    all_pred,
    all_gt,
    all_participant_task_chunk_list,
    fs,
    ll_cutoff=40,
    ul_cutoff=180
):
    all_pred_hr = []
    all_gt_hr = []
    all_goodness = []
    all_pred_ibi = []
    all_gt_ibi = []

    for pred, gt, (participant_task, chunk_id_list) in zip(all_pred, all_gt, all_participant_task_chunk_list):
        #print(participant_task, chunk_id_list)

        # Normalize
        pred = (pred - np.mean(pred)) / np.std(pred)
        gt = (gt - np.mean(gt)) / np.std(gt)

        # Detrend and Goodness
        goodness = calcMetrics(pred, fs, 0.1, normalize=True)[1][0]

        # 1-D gauss blur
        # pred = custom_detrend(pred, 100)
        # gt = custom_detrend(gt, 100)
        pred = np.convolve(pred, np.ones((7)) / 7, mode='same')
        gt = np.convolve(gt, np.ones((7)) / 7, mode='same')

        # Re-normalize
        pred = (pred - np.mean(pred)) / np.std(pred)
        gt = (gt - np.mean(gt)) / np.std(gt)

        # Heart rate via PSD
        pred_hr = pulse_rate_from_power_spectral_density(
            pred, fs, ll_cutoff, ul_cutoff, BUTTER_ORDER=6, DETREND=False, WELCH=True)
        gt_hr = pulse_rate_from_power_spectral_density(
            gt, fs, ll_cutoff, ul_cutoff, BUTTER_ORDER=6, DETREND=False, WELCH=True)

        all_pred_hr.append(pred_hr)
        all_gt_hr.append(gt_hr)
        all_goodness.append(goodness)

        #print(f"| {pred_hr:.2f} - {gt_hr:.2f} | = {abs(pred_hr - gt_hr):.2f} bpm\t\t\t{goodness:.2f}")

        # Inter-beat interval
        pred_ibi, gt_ibi = get_filtered_ibi(pred, gt, fs=fs, smooth_window=7)
        all_pred_ibi.append(np.mean(pred_ibi))
        all_gt_ibi.append(np.mean(gt_ibi))

        #print(f"IBI: | {np.mean(pred_ibi):.3f} - {np.mean(gt_ibi):.3f} | = {abs(np.mean(pred_ibi) - np.mean(gt_ibi)):.3f} secs")
        #print("-" * 100)

    return (
        np.array(all_pred_hr),
        np.array(all_gt_hr),
        np.array(all_goodness),
        np.array(all_pred_ibi),
        np.array(all_gt_ibi)
    )
    
    
def get_error_metric(pred_values, gt_values):
    """
    Calculate the error metric between predicted and ground truth values.
    """
    # Calculate the mean absolute error
    mae = np.mean(np.abs(pred_values - gt_values))
    # Calculate the root mean squared error
    rmse = np.sqrt(np.mean(np.square(np.abs(pred_values - gt_values))))
    # Calculate the mean absolute percentage error
    mape = np.mean(np.abs((pred_values - gt_values) / gt_values)) * 100
    # Calculate pearson correlation coefficient
    r = np.corrcoef(pred_values, gt_values)[0, 1]
    return mae, rmse, mape, r


def evaluate_and_log_metrics(all_pred_hr, all_gt_hr, all_goodness, all_pred_ibi, all_gt_ibi, epoch_num, plot=False):

    all_errors = np.abs(all_pred_hr - all_gt_hr)

    # Error metrics
    mae, rmse, mape, r = get_error_metric(all_pred_hr, all_gt_hr)
    ibi_error = np.mean(np.abs(all_pred_ibi - all_gt_ibi)) * 1000  # in ms

    # Print LaTeX-style row
    print("MAE, RMSE, MAPE, r, IBI")
    print(np.round(mae,2), "&", np.round(rmse,2), "&", np.round(mape,2), "&", np.round(r,2), "&", np.round(ibi_error,2), r"\\")

    # Plot results
    if plot:
        plt.figure(figsize=(20, 8))
        plt.subplot(1, 2, 1)
        plt.plot(all_pred_hr, all_gt_hr, 'o', alpha=0.5)
        plt.xlabel('Predicted HR')
        plt.ylabel('Ground Truth HR')
        plt.title(f'HR Prediction (Epoch {epoch_num})') 

        plt.subplot(1, 2, 2)
        plt.plot(all_goodness, all_errors, 'o', alpha=0.5)
        plt.xlabel('Goodness')
        plt.ylabel('Absolute HR Error')
        plt.title(f'Goodness vs HR Error (Epoch {epoch_num})')
        plt.show()

        plt.figure(figsize=(15, 5))
        plt.hist(np.array(all_pred_hr), bins=100, alpha=0.5, label='Predicted HR')
        plt.hist(np.array(all_gt_hr), bins=100, alpha=0.5, label='GT HR')
        plt.legend()
        plt.title(f'HR Distribution (Epoch {epoch_num})')
        plt.show()

    return np.array([mae, rmse, mape, r, ibi_error])



In [ ]:
metrics_table = []

all_epochs_load_path = '/home/ab227/CogPhys/runs/exp/fusion_rppg_early_mid_pre_nir/PreTrainedModels'
for i in range(0, len(os.listdir(all_epochs_load_path))):
    epoch_num = 5 * i - 1
    if epoch_num < 0:
        epoch_num = 0
    if epoch_num >= len(os.listdir(all_epochs_load_path)):
        break
    
    model = load_one_path(all_epochs_load_path, epoch_num)

    all_pred = []
    all_gt = []
    all_participant_task_chunk_list = []
    for i in range(0, len(test_data_loader), 12):
        for j in [[0, 1, 4], [5, 6, 7], [8, 9, 10], [11, 2, 3]]:
            img = []
            label = []
            participant_task_list = []
            chunk_id_list = []
            with torch.no_grad():
                for k in j:
                    img_sample, label_sample, participant_task, chunk_id = test_data_loader[i+k]
                    participant_task_list.append(participant_task)
                    chunk_id_list.append(int(chunk_id))
                    img.append(img_sample.to(config.DEVICE))
                    label.extend(label_sample.squeeze(0).cpu().numpy().tolist())
                img = torch.cat(img, dim=1).unsqueeze(0)
                pred = forward_pass(model, img)
            if "v23_read" == participant_task:
                print("Skipping")
                continue
            ##################
            assert participant_task_list[0] == participant_task_list[1]
            assert chunk_id_list[0] == chunk_id_list[1]-1
            ##################
            pred = np.array(pred)
            label = np.array(label)
            all_participant_task_chunk_list.append((participant_task_list[0], chunk_id_list)) 
            all_pred.append(pred)
            all_gt.append(label)
            print(f'epoch{epoch_num}', participant_task_list[0], chunk_id_list)
            
    #now calc metrics for this model epoch path
    all_pred_hr, all_gt_hr, all_goodness, all_pred_ibi, all_gt_ibi = process_hr_ibi(all_pred, all_gt, all_participant_task_chunk_list, fs = config.TRAIN.DATA.FS)
    metrics = evaluate_and_log_metrics(all_pred_hr, all_gt_hr, all_goodness, all_pred_ibi, all_gt_ibi, epoch_num)
    print(metrics)
    metrics_table.append((epoch_num, metrics))
    
print(metrics_table)

epoch0 v12_number [0, 1, 2]
epoch0 v12_number [3, 4, 5]
epoch0 v12_number [6, 7, 8]
epoch0 v12_number [9, 10, 11]
epoch0 v12_still [0, 1, 2]
epoch0 v12_still [3, 4, 5]
epoch0 v12_still [6, 7, 8]
epoch0 v12_still [9, 10, 11]
epoch0 v33_read_rest [0, 1, 2]
epoch0 v33_read_rest [3, 4, 5]
epoch0 v33_read_rest [6, 7, 8]
epoch0 v33_read_rest [9, 10, 11]
epoch0 v12_read [0, 1, 2]
epoch0 v12_read [3, 4, 5]
epoch0 v12_read [6, 7, 8]
epoch0 v12_read [9, 10, 11]
epoch0 v33_pattern_rest [0, 1, 2]
epoch0 v33_pattern_rest [3, 4, 5]
epoch0 v33_pattern_rest [6, 7, 8]
epoch0 v33_pattern_rest [9, 10, 11]
epoch0 v33_read [0, 1, 2]
epoch0 v33_read [3, 4, 5]
epoch0 v33_read [6, 7, 8]
epoch0 v33_read [9, 10, 11]
epoch0 v12_pattern [0, 1, 2]
epoch0 v12_pattern [3, 4, 5]
epoch0 v12_pattern [6, 7, 8]
epoch0 v12_pattern [9, 10, 11]
epoch0 v33_still [0, 1, 2]
epoch0 v33_still [3, 4, 5]
epoch0 v33_still [6, 7, 8]
epoch0 v33_still [9, 10, 11]
epoch0 v12_pattern_rest [0, 1, 2]
epoch0 v12_pattern_rest [3, 4, 5]
epoc

In [2]:
import pandas as pd
import os
import numpy as np

#np.save('val_epoch_compare/rppg_contrastfusion_test.npy', np.stack([row[1] for row in metrics_table]))
loaded_metrics = np.load('val_epoch_compare/rppg_contrastfusion_test.npy')  # shape (N, num_metrics)
all_epochs_load_path = '/home/ab227/CogPhys/runs/exp/fusion_rppg_early_mid_pre_nir/PreTrainedModels'
epoch_nums = []
for i in range(0, len(os.listdir(all_epochs_load_path))):
    epoch_num = 5 * i - 1
    if epoch_num < 0:
        epoch_num = 0
    if epoch_num >= len(os.listdir(all_epochs_load_path)):
        break
    epoch_nums.append(epoch_num)
metrics_table = [[epoch, row] for epoch, row in zip(epoch_nums, loaded_metrics)]

# Convert to DataFrame
df = pd.DataFrame(metrics_table, columns=["Epoch", "Metrics"])
df[['MAE', 'RMSE', 'MAPE', 'r', 'IBI']] = pd.DataFrame(df["Metrics"].tolist(), index=df.index)
df = df.drop(columns="Metrics")

# Optional: format float display
pd.options.display.float_format = '{:.4f}'.format

# Show the formatted table
print(df.to_string(index=False))


 Epoch    MAE   RMSE   MAPE      r     IBI
     0 3.9148 7.0382 4.6129 0.8150 39.1397
     4 3.7097 6.4695 4.3733 0.8476 40.9146
     9 2.8881 4.9460 3.5036 0.9073 32.8448
    14 3.2258 5.4745 3.9108 0.8867 36.3889
    19 3.0373 5.1399 3.7700 0.8984 29.2374
    24 3.1856 5.5563 3.8090 0.8839 34.6023
    29 3.0165 5.0054 3.6483 0.9064 33.4809
    34 2.9102 4.8307 3.5530 0.9124 30.1222
    39 3.2102 5.5586 3.8609 0.8836 33.8904
    44 2.9042 4.6447 3.6323 0.9192 31.3199
    49 2.9008 4.8720 3.5284 0.9114 35.2615
    54 3.3966 5.9363 4.0051 0.8741 37.8751
    59 4.1606 7.5446 4.8319 0.7958 47.3948
    64 3.7407 6.8120 4.3832 0.8283 39.7425


In [11]:
# print(len(all_pred), len(all_gt), len(all_participant_task_chunk_list))
# save_folder = "waveforms/fusion/"
# os.makedirs(save_folder, exist_ok=True)
# with open(os.path.join(save_folder, "pred.pickle"), 'wb') as f:
#     pickle.dump({'pred': all_pred, 'gt': all_gt, 'participant_task_chunk_id_list': all_participant_task_chunk_list}, f)
# with open(os.path.join(save_folder, "pred.pickle"), 'rb') as f:
#     data = pickle.load(f)
#     pred = data['pred']
#     gt = data['gt']
#     participant_task_chunk_id_list = data['participant_task_chunk_id_list']
# print(len(pred), len(gt), len(participant_task_chunk_id_list))